# 🧹 Notebook 02 — Tratamento de Dados

---

## Sobre este notebook

Dados brutos raramente chegam prontos para análise. Nessa etapa, vamos fazer a limpeza e a preparação dos dados coletados no notebook anterior.

O tratamento aqui é intencional e simples — o objetivo não é fazer transformações sofisticadas, mas sim garantir que os dados estejam:

- Sem valores ausentes que possam causar erros nas análises
- Com os tipos de dados corretos (datas como `datetime`, por exemplo)
- Contendo apenas as colunas que realmente vamos usar
- Com uma nova métrica calculada: a **variação percentual diária**

### O que vamos fazer

1. Ler os arquivos CSV brutos
2. Remover linhas com valores nulos
3. Garantir que a coluna de datas está no tipo correto
4. Selecionar apenas as colunas `Open`, `Close` e `Volume`
5. Criar a coluna `variacao_percentual_diaria`
6. Salvar os dados tratados em `data/processed/`

---

## 1. Importação das bibliotecas

In [1]:
import pandas as pd
import numpy as np
import os

---

## 2. Leitura dos arquivos brutos

Vamos carregar os três CSVs salvos na etapa anterior. O parâmetro `index_col=0` é necessário porque o yfinance salva a coluna de data como índice, e o pandas precisa saber disso na hora de ler.

In [2]:
# Lendo os arquivos brutos de cada ativo
dados_petr4_bruto = pd.read_csv("../data/raw/petr4_dados_brutos.csv", index_col=0, header=[0, 1])
dados_vale3_bruto = pd.read_csv("../data/raw/vale3_dados_brutos.csv", index_col=0, header=[0, 1])
dados_itub4_bruto = pd.read_csv("../data/raw/itub4_dados_brutos.csv", index_col=0, header=[0, 1])

print("✅ Arquivos lidos com sucesso!")
print(f"\nPETR4: {dados_petr4_bruto.shape[0]} linhas, {dados_petr4_bruto.shape[1]} colunas")
print(f"VALE3: {dados_vale3_bruto.shape[0]} linhas, {dados_vale3_bruto.shape[1]} colunas")
print(f"ITUB4: {dados_itub4_bruto.shape[0]} linhas, {dados_itub4_bruto.shape[1]} colunas")

✅ Arquivos lidos com sucesso!

PETR4: 251 linhas, 5 colunas
VALE3: 251 linhas, 5 colunas
ITUB4: 251 linhas, 5 colunas


---

## 3. Simplificação do MultiIndex de colunas

O yfinance retorna um DataFrame com MultiIndex nas colunas (nome da métrica + ticker). Vamos simplificar isso para trabalhar mais facilmente.

In [3]:
def simplificar_colunas_yfinance(dataframe):
    """
    O yfinance retorna colunas em formato MultiIndex (ex: ('Close', 'PETR4.SA')).
    Essa função extrai só o primeiro nível, deixando apenas o nome da métrica.
    """
    # Se as colunas tiverem múltiplos níveis, pegamos só o primeiro
    if isinstance(dataframe.columns, pd.MultiIndex):
        dataframe.columns = dataframe.columns.get_level_values(0)
    return dataframe


dados_petr4_bruto = simplificar_colunas_yfinance(dados_petr4_bruto)
dados_vale3_bruto = simplificar_colunas_yfinance(dados_vale3_bruto)
dados_itub4_bruto = simplificar_colunas_yfinance(dados_itub4_bruto)

print("Colunas disponíveis (PETR4):")
print(dados_petr4_bruto.columns.tolist())

Colunas disponíveis (PETR4):
['Close', 'High', 'Low', 'Open', 'Volume']


---

## 4. Tratamento de valores nulos

Valores nulos em séries temporais financeiras geralmente surgem em datas onde o mercado estava fechado ou houve alguma inconsistência na coleta. Para análises gráficas, eles causam lacunas e erros — então a decisão mais segura é removê-los.

In [4]:
# Verificando a quantidade de nulos antes de remover
print("Valores nulos antes do tratamento:")
print(f"PETR4: {dados_petr4_bruto.isnull().sum().sum()} valores nulos")
print(f"VALE3: {dados_vale3_bruto.isnull().sum().sum()} valores nulos")
print(f"ITUB4: {dados_itub4_bruto.isnull().sum().sum()} valores nulos")

Valores nulos antes do tratamento:
PETR4: 0 valores nulos
VALE3: 0 valores nulos
ITUB4: 0 valores nulos


In [5]:
# Removendo as linhas com pelo menos um valor nulo
# Usamos dropna() em vez de preenchimento para não distorcer os dados financeiros
dados_petr4_sem_nulos = dados_petr4_bruto.dropna()
dados_vale3_sem_nulos = dados_vale3_bruto.dropna()
dados_itub4_sem_nulos = dados_itub4_bruto.dropna()

print("Linhas removidas por conterem nulos:")
print(f"PETR4: {len(dados_petr4_bruto) - len(dados_petr4_sem_nulos)} linhas")
print(f"VALE3: {len(dados_vale3_bruto) - len(dados_vale3_sem_nulos)} linhas")
print(f"ITUB4: {len(dados_itub4_bruto) - len(dados_itub4_sem_nulos)} linhas")

Linhas removidas por conterem nulos:
PETR4: 0 linhas
VALE3: 0 linhas
ITUB4: 0 linhas


---

## 5. Conversão da coluna de datas

O índice do DataFrame veio como string depois da leitura do CSV. Precisamos converter para `datetime` para que o pandas consiga ordenar, filtrar e plotar corretamente por data.

In [6]:
def converter_indice_para_datetime(dataframe):
    """
    Garante que o índice do DataFrame seja do tipo datetime.
    Isso é importante para que os gráficos temporais fiquem com o eixo X correto.
    """
    dataframe.index = pd.to_datetime(dataframe.index)
    return dataframe


dados_petr4_sem_nulos = converter_indice_para_datetime(dados_petr4_sem_nulos)
dados_vale3_sem_nulos = converter_indice_para_datetime(dados_vale3_sem_nulos)
dados_itub4_sem_nulos = converter_indice_para_datetime(dados_itub4_sem_nulos)

print(f"Tipo do índice (PETR4): {dados_petr4_sem_nulos.index.dtype}")

Tipo do índice (PETR4): datetime64[us]


---

## 6. Seleção das colunas principais

O yfinance retorna várias colunas (`Open`, `High`, `Low`, `Close`, `Adj Close`, `Volume`). Para este projeto, vamos trabalhar apenas com:

- **Open** — preço de abertura
- **Close** — preço de fechamento
- **Volume** — volume negociado no dia

Essa seleção deixa os dados mais enxutos e facilita a leitura nas etapas seguintes.

In [7]:
# Colunas que vamos manter para as análises
colunas_selecionadas = ["Open", "Close", "Volume"]

dados_petr4_colunas_selecionadas = dados_petr4_sem_nulos[colunas_selecionadas].copy()
dados_vale3_colunas_selecionadas = dados_vale3_sem_nulos[colunas_selecionadas].copy()
dados_itub4_colunas_selecionadas = dados_itub4_sem_nulos[colunas_selecionadas].copy()

print("Colunas após seleção:")
print(dados_petr4_colunas_selecionadas.columns.tolist())
print(dados_petr4_colunas_selecionadas.head(3))

Colunas após seleção:
['Open', 'Close', 'Volume']
Price            Open     Close    Volume
Date                                     
2024-01-02  27.065107  27.31089  24043800
2024-01-03  27.325345  28.16390  52300200
2024-01-04  28.279566  27.92535  45344900


---

## 7. Criação da coluna de variação percentual diária

Essa é a única transformação nova que vamos criar. A variação percentual diária mostra o quanto o preço do ativo subiu ou caiu em cada dia, em relação à sua abertura.

**Fórmula:**

$$
\text{variacao\_percentual\_diaria} = \frac{\text{Close} - \text{Open}}{\text{Open}} \times 100
$$

Valores positivos indicam que o ativo fechou acima da abertura. Valores negativos, abaixo.

In [8]:
def calcular_variacao_percentual_diaria(dataframe):
    """
    Cria uma nova coluna com a variação percentual entre abertura e fechamento.
    Essa métrica é útil para entender a volatilidade intradiária do ativo.
    """
    dataframe["variacao_percentual_diaria"] = (
        (dataframe["Close"] - dataframe["Open"]) / dataframe["Open"]
    ) * 100
    return dataframe


dados_petr4_tratado = calcular_variacao_percentual_diaria(dados_petr4_colunas_selecionadas)
dados_vale3_tratado = calcular_variacao_percentual_diaria(dados_vale3_colunas_selecionadas)
dados_itub4_tratado = calcular_variacao_percentual_diaria(dados_itub4_colunas_selecionadas)

print("Amostra dos dados tratados com a nova coluna:")
print(dados_petr4_tratado.head())

Amostra dos dados tratados com a nova coluna:
Price            Open      Close    Volume  variacao_percentual_diaria
Date                                                                  
2024-01-02  27.065107  27.310890  24043800                    0.908120
2024-01-03  27.325345  28.163900  52300200                    3.068783
2024-01-04  28.279566  27.925350  45344900                   -1.252551
2024-01-05  28.098842  27.990410  35783700                   -0.385896
2024-01-08  27.744621  27.780766  35158100                    0.130274


Vamos também checar algumas estatísticas da nova coluna para ver se os valores fazem sentido:

In [9]:
print("Estatísticas da variação percentual diária:")
print("\nPETR4:")
print(dados_petr4_tratado["variacao_percentual_diaria"].describe().round(2))

print("\nVALE3:")
print(dados_vale3_tratado["variacao_percentual_diaria"].describe().round(2))

print("\nITUB4:")
print(dados_itub4_tratado["variacao_percentual_diaria"].describe().round(2))

Estatísticas da variação percentual diária:

PETR4:
count    251.00
mean       0.01
std        1.32
min       -4.92
25%       -0.83
50%       -0.05
75%        0.85
max        4.52
Name: variacao_percentual_diaria, dtype: float64

VALE3:
count    251.00
mean      -0.16
std        1.01
min       -2.77
25%       -0.76
50%       -0.13
75%        0.50
max        3.06
Name: variacao_percentual_diaria, dtype: float64

ITUB4:
count    251.00
mean      -0.04
std        1.04
min       -3.30
25%       -0.70
50%       -0.06
75%        0.62
max        3.67
Name: variacao_percentual_diaria, dtype: float64


---

## 8. Salvamento dos dados tratados

Com o tratamento concluído, vamos salvar os arquivos na pasta `data/processed/`. A separação entre dados brutos e dados processados é importante — se precisarmos refazer alguma transformação, o arquivo original continua intacto.

In [10]:
# Criando a pasta de dados processados, caso ainda não exista
os.makedirs("../data/processed", exist_ok=True)

# Caminhos de destino para os arquivos tratados
caminho_arquivo_petr4_tratado = "../data/processed/petr4_dados_tratados.csv"
caminho_arquivo_vale3_tratado = "../data/processed/vale3_dados_tratados.csv"
caminho_arquivo_itub4_tratado = "../data/processed/itub4_dados_tratados.csv"

# Salvando cada arquivo
dados_petr4_tratado.to_csv(caminho_arquivo_petr4_tratado)
dados_vale3_tratado.to_csv(caminho_arquivo_vale3_tratado)
dados_itub4_tratado.to_csv(caminho_arquivo_itub4_tratado)

print("✅ Dados tratados salvos com sucesso em data/processed/:")
print(f"   → {caminho_arquivo_petr4_tratado}")
print(f"   → {caminho_arquivo_vale3_tratado}")
print(f"   → {caminho_arquivo_itub4_tratado}")

✅ Dados tratados salvos com sucesso em data/processed/:
   → ../data/processed/petr4_dados_tratados.csv
   → ../data/processed/vale3_dados_tratados.csv
   → ../data/processed/itub4_dados_tratados.csv


---

## ✅ Conclusão desta etapa

O tratamento foi simples, mas cobriu os pontos essenciais:

| Transformação | O que foi feito |
|---|---|
| Remoção de nulos | Linhas incompletas foram excluídas para evitar erros nas análises |
| Conversão de datas | O índice foi convertido para `datetime` para facilitar plotagens |
| Seleção de colunas | Mantivemos apenas `Open`, `Close` e `Volume` |
| Nova coluna | Calculamos a `variacao_percentual_diaria` para cada ativo |

### Próximos passos

No **Notebook 03**, vamos usar esses dados tratados para criar três visualizações que ajudam a entender o comportamento de cada ativo ao longo do período analisado.

> 💡 A etapa de tratamento pode parecer simples ou chata, mas ela é uma das mais importantes do pipeline. Dados sujos ou mal formatados comprometem qualquer análise, por mais sofisticada que seja.